# Hybrid Architectures: Building Mixed Models

Build a hybrid SSM/attention model step by step using
`block_configs`. Visualize which layers are SSM vs attention
and compare their behavior.

**Prerequisites:** `uv sync --extra dev --extra analysis`

In [ ]:
import mlx.core as mx

from lmxlab.core.config import BlockConfig, ModelConfig
from lmxlab.models.base import LanguageModel

## Per-Layer Block Configs

lmxlab's `ModelConfig` supports `block_configs`: a tuple of
`BlockConfig` objects, one per layer. This lets you mix
attention and SSM layers freely.

In [ ]:
# Define an attention block config
attn_block = BlockConfig(
    attention="gqa",
    ffn="gated",
    norm="rms_norm",
    position="rope",
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
)

# Define a Mamba-2 SSM block config
ssm_block = BlockConfig(
    attention="mamba2",
    ffn="gated",
    norm="rms_norm",
    position="none",  # SSM layers don't use position encoding
    d_model=64,
    n_heads=4,
    n_kv_heads=2,
    d_ff=128,
    bias=False,
    max_seq_len=128,
    mamba_n_heads=4,
    mamba_head_dim=32,
    ssm_state_size=64,
    mamba_expand=2,
)

print("Attention block:")
print(f"  attention={attn_block.attention}, position={attn_block.position}")
print(f"\nSSM block:")
print(f"  attention={ssm_block.attention}, position={ssm_block.position}")
print(f"  mamba_n_heads={ssm_block.mamba_n_heads}, mamba_head_dim={ssm_block.mamba_head_dim}")

## Build a Custom Hybrid Model

Let's create a 6-layer hybrid with pattern `MMMAMM`:
mostly SSM with one attention layer in the middle.

In [ ]:
pattern = "MMMAMM"  # M=Mamba, A=Attention

blocks = tuple(
    attn_block if c == "A" else ssm_block
    for c in pattern
)

hybrid_config = ModelConfig(
    block=attn_block,  # Default block (used for final norm etc.)
    vocab_size=256,
    n_layers=len(pattern),
    block_configs=blocks,
    tie_embeddings=True,
)

print(f"Hybrid model: {len(pattern)} layers")
for i, c in enumerate(pattern):
    layer_type = "Attention (GQA)" if c == "A" else "SSM (Mamba-2)"
    print(f"  Layer {i}: {layer_type}")

In [ ]:
hybrid = LanguageModel(hybrid_config)
mx.eval(hybrid.parameters())

print(f"Total params: {hybrid.count_parameters():,}")

tokens = mx.zeros((1, 16), dtype=mx.int32)
logits, caches = hybrid(tokens)
mx.eval(logits)
print(f"Output: {logits.shape}")

## Compare with Pre-Built Hybrids

lmxlab provides factory functions for published hybrid designs.

In [ ]:
from lmxlab.models.falcon import falcon_h1_tiny
from lmxlab.models.jamba import jamba_tiny
from lmxlab.models.bamba import bamba_tiny

for name, factory in [("Falcon-H1", falcon_h1_tiny), ("Jamba", jamba_tiny), ("Bamba", bamba_tiny)]:
    cfg = factory()
    m = LanguageModel(cfg)
    mx.eval(m.parameters())
    n_ssm = sum(1 for i in range(cfg.n_layers) if "mamba" in cfg.get_block_config(i).attention)
    n_attn = cfg.n_layers - n_ssm
    print(f"{name}: {m.count_parameters():,} params, {n_ssm} SSM + {n_attn} attention layers")

## Activation Comparison: SSM vs Attention Layers

Use ActivationCapture to compare how SSM and attention
layers transform the residual stream.

In [ ]:
from lmxlab.analysis.activations import ActivationCapture

with ActivationCapture(hybrid) as cap:
    hybrid(tokens)

norms = cap.layer_norms()

print(f"{'Layer':>5} {'Type':>6} {'Input':>8} {'Output':>8}")
print("-" * 30)
for i, c in enumerate(pattern):
    ltype = "Attn" if c == "A" else "SSM"
    in_norm = norms.get(f"layer_{i}/input", 0)
    out_norm = norms.get(f"layer_{i}/output", 0)
    print(f"{i:>5} {ltype:>6} {in_norm:>8.3f} {out_norm:>8.3f}")